# Isolated V2 Mem adapter: prepare data directly on Kaggle
Attach **only `mem_adapter_v1_code.zip`** to start. Enable Internet and select two T4 GPUs.
Default: identify pinned sources -> download/reuse raw data -> verify hashes -> predict
causal T1/T3 memory -> build mixed shards -> validate -> train the unmerged adapter.
No local data ZIP is required. Existing completed datasets remain supported via
`DATA_MODE = 'attached_prepared'`. This is a custom V2 experiment, not an official LIBERO score.

Raw inputs total 42.54 GB (39.62 GiB), before shards, environments and models.
Scratch files in `/kaggle/temp` are session-local. Memory indices/reports and training
outputs go to `/kaggle/working`; check their combined size before saving a version.
Kaggle documents a 20 GB saved-output limit: [Notebook docs](https://www.kaggle.com/docs/notebooks).
Memory generation uses the existing sequential writer on one GPU; two-GPU training
starts only after preparation completes. Runtime has not been measured on Kaggle.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

DATA_MODE = 'prepare_on_kaggle'  # or 'attached_prepared'; never an automatic fallback
if DATA_MODE not in {'prepare_on_kaggle', 'attached_prepared'}:
    raise ValueError('Unknown DATA_MODE')
INPUT = Path('/kaggle/input')
SCRATCH = Path('/kaggle/temp/mem-adapter-v1')
WORK = Path('/kaggle/working/mem-adapter-v1')
SCRATCH.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
REPO = SCRATCH / 'code'
PREP = WORK / 'preparation'
ASSETS = SCRATCH / 'assets'
TRAIN_PY = SCRATCH / 'training-env/bin/python'
PERCEPTION_PY = SCRATCH / 'perception-env/bin/python'
# Planning allowance, NOT a measured upper bound for processed data or peak disk.
PROCESSING_RESERVE_GIB = 60

def run(python, *args):
    subprocess.run([str(python), *map(str, args)], cwd=REPO, check=True)

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Archive path escapes destination')
        z.extractall(destination)

code_zips = list(INPUT.rglob('mem_adapter_v1_code.zip'))
if len(code_zips) != 1:
    raise ValueError('Attach exactly one updated mem_adapter_v1_code.zip')
# Refuse a changed bundle in the same scratch root, rather than mixing versions.
code_hash = digest(code_zips[0])
receipt = WORK / 'code.sha256'
if receipt.exists() and receipt.read_text().strip() != code_hash:
    raise ValueError('Code bundle changed; use a fresh run root')
extract_checked(code_zips[0], REPO)
receipt.write_text(code_hash + '\n')
print('Data mode:', DATA_MODE, '| Code SHA256:', code_hash)


## 1. Identify sources and available disk before downloading
The plan lists author repository, immutable revision, expected bytes/SHA256 and origin
(`existing`, raw Kaggle `attached`, or `download`). Only the 10 reviewed base files,
T1/T3 and metainfo are selected. Name/size matching is only discovery; full hashes are
verified before reuse. Duplicate or incomplete attached files cause an error.


In [ ]:
if DATA_MODE == 'prepare_on_kaggle':
    run(sys.executable, REPO / 'scripts/download_memory_training_sources.py',
        '--input-root', INPUT, '--plan-only', '--report', WORK / 'source-plan.json')
    plan = json.loads((WORK / 'source-plan.json').read_text())
    required = plan['download_bytes'] + PROCESSING_RESERVE_GIB * 2**30
    free = shutil.disk_usage(SCRATCH).free
    print(json.dumps({'source_GiB': plan['total_source_bytes'] / 2**30,
                      'download_GiB': plan['download_bytes'] / 2**30,
                      'free_scratch_GiB': free / 2**30,
                      'additional_processing_allowance_GiB': PROCESSING_RESERVE_GIB}, indent=2))
    if free < required:
        raise RuntimeError('Insufficient scratch disk for downloads + planning allowance; inspect source-plan.json')
    locked = json.loads((REPO / 'resources/manifests/libero_mem_t1_t3_base10_split_v1.json').read_text())
    from collections import Counter
    print('Locked episodes:', len(locked['episodes']), Counter(e['split'] for e in locked['episodes']))
else:
    print('Using an explicitly attached completed dataset; source downloads are skipped.')


## 2. Install isolated environments
Perception keeps Transformers 4.51.3 and its reviewed detector dependencies.
Training uses its own pinned recipe. No packages are installed into the notebook kernel.
Both environments run in fresh subprocesses and record `pip freeze`.


In [ ]:
def install_env(python, requirements):
    if not python.is_file():
        run(sys.executable, '-m', 'venv', python.parent.parent)
    run(python, '-m', 'pip', 'install', '--no-cache-dir', '-r', REPO / requirements)
    frozen = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], text=True)
    (WORK / (python.parent.parent.name + '-pip-freeze.txt')).write_text(frozen)

install_env(TRAIN_PY, 'envs/mem_adapter_training.txt')
run(TRAIN_PY, '-c', 'import torch; assert torch.cuda.device_count() == 2; print(torch.__version__); print([torch.cuda.get_device_name(i) for i in range(2)])')
if DATA_MODE == 'prepare_on_kaggle':
    install_env(PERCEPTION_PY, 'envs/memory_perception.txt')
    run(PERCEPTION_PY, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))')


In [ ]:
# Fetch the original policy and tokenizer at the repository's pinned revisions.
run(TRAIN_PY, REPO / 'scripts/memory_sft_assets.py', 'fetch', ASSETS)
if DATA_MODE == 'prepare_on_kaggle':
    run(PERCEPTION_PY, REPO / 'scripts/download_memory_training_sources.py',
        '--input-root', INPUT, '--report', WORK / 'source-verified.json')
    run(PERCEPTION_PY, REPO / 'scripts/download_memory_detector.py',
        '--output', REPO / 'resources/models/grounding-dino-tiny')


## 3. Predict memory and build the dataset, or use an attached completed dataset
Reuse the exact 698-demo split lock (558/70/70). Memory generation keeps the existing
causal writer, camera convention and tokenizer. The mixed dataset retains all locked
episodes; **50/50 is the training sampler ratio**, not an instruction to discard half
the data. Completed stages can be reused; partial stage directories deliberately fail.
Inspect and quarantine an incomplete stage before retrying; no automatic deletion.
To reuse indices from a previous saved notebook, attach that output and copy its
completed `memory_T1` / `memory_T3` directories into `PREP` before running this cell.
Their hashes, split and writer contracts are checked when building the mixed dataset.


In [ ]:
if DATA_MODE == 'prepare_on_kaggle':
    PREP.mkdir(parents=True, exist_ok=True)
    source_lock = REPO / 'resources/manifests/libero_mem_t1_t3_base10_split_v1.json'
    output_lock = PREP / 'split.lock.json'
    if output_lock.exists():
        if digest(output_lock) != digest(source_lock):
            raise ValueError('Existing split differs from the bundled expanded lock')
    else:
        shutil.copy2(source_lock, output_lock)
    DATA = SCRATCH / 'dataset'
    run(PERCEPTION_PY, REPO / 'scripts/prepare_mem_adapter_data.py',
        '--output-root', PREP, '--dataset-dir', DATA,
        '--perception-python', PERCEPTION_PY, '--tokenizer-path', ASSETS / 'backbone')
else:
    data_candidates = []
    for p in INPUT.rglob('manifest.json'):
        try:
            m = json.loads(p.read_text())
        except (ValueError, OSError):
            continue
        if (m.get('artifact') == 'mixed_base_mem_sft_shards'
                and m.get('memory_source') == 'predicted_rgb_v3'
                and len(m.get('retention_replay_tasks', [])) == 10):
            data_candidates.append(p.parent)
    archives = list(INPUT.rglob('mem_adapter_v1_data.zip'))
    if len(data_candidates) + len(archives) != 1:
        raise ValueError('Attach exactly one completed expanded dataset directory OR data ZIP')
    if data_candidates:
        DATA = data_candidates[0]
    else:
        sidecar = archives[0].with_suffix('.json')
        if not sidecar.is_file() or digest(archives[0]) != json.loads(sidecar.read_text())['sha256']:
            raise ValueError('Data ZIP SHA256 sidecar missing or mismatched')
        with zipfile.ZipFile(archives[0]) as z:
            if sum(m.file_size for m in z.infolist()) > shutil.disk_usage(SCRATCH).free:
                raise RuntimeError('Insufficient disk to extract the attached data ZIP')
        extract_checked(archives[0], SCRATCH / 'attached-data')
        DATA = SCRATCH / 'attached-data/dataset'
run(TRAIN_PY, REPO / 'scripts/validate_mixed_memory_sft.py', '--data-dir', DATA)
shutil.copy2(DATA / 'manifest.json', WORK / 'dataset-manifest.json')
shutil.copy2(DATA / 'split.lock.json', WORK / 'dataset-split.lock.json')
print('Validated dataset:', DATA)


## 4. Train the unmerged adapter
Raw data and large shards remain in scratch and are consumed directly, without creating
a data ZIP. Preserve completed memory indices to avoid repeating detector work. Scratch
data must be downloaded/rebuilt in a later session unless separately persisted on Kaggle.


In [ ]:
from datetime import datetime, timezone
OUTPUT = Path('/kaggle/working') / ('mem-adapter-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
run(TRAIN_PY, REPO / 'scripts/train_mem_adapter.py', '--data-dir', DATA,
    '--assets-dir', ASSETS, '--output-dir', OUTPUT, '--gpus', '2')
print('Unmerged adapter:', OUTPUT / 'pretrained_model')
print('Training export complete; policy success remains unmeasured.')
saved_bytes = sum(p.stat().st_size for p in Path('/kaggle/working').rglob('*') if p.is_file())
print('Working output GB:', saved_bytes / 1e9)


Save the output version and retain metrics, source/code receipts, split lock, completed
memory indices, run config, checkpoints and `pretrained_model` within the account's output
quota. The data ZIP is optional and is not created by this notebook. Save only completed
stages; a cancelled session does not establish successful preparation or training.
Use the original base checkpoint for baseline tasks. Deploy the adapter through the F1
runner with its verified writer and pinned tokenizer. The final checkpoint is not
automatically the best checkpoint. This notebook has been checked locally for structure
and contracts; end-to-end Kaggle execution is still required.
